# 19.1 Interpretability: feature importance & SHAP

Feature importance ranks what the model used; SHAP goes further by assigning each feature a signed share of one prediction.

We replace a single score with a signed receipt against a baseline. The pitfall is treating global gain or absolute ranks as a local explanation.

Save a copy to Drive to edit.

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(19)


def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def fit_ladder_classifier(X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr_s = scaler.fit_transform(x_tr)
    x_te_s = scaler.transform(x_te)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr_s, y_tr)
    return clf, scaler, x_tr_s, x_te_s, y_tr, y_te


def score_for_class(model, X, class_index=None):
    proba = model.predict_proba(X)
    if class_index is None:
        class_index = int(np.argmax(proba[0]))
    return proba[:, class_index]


def finite_gradient(fn, x, eps=1e-4):
    grad = np.zeros_like(x, dtype=float)
    for j in range(len(x)):
        plus = x.copy()
        minus = x.copy()
        plus[j] = plus[j] + eps
        minus[j] = minus[j] - eps
        grad[j] = (fn(plus) - fn(minus)) / (2.0 * eps)
    return grad


def rank_corr(a, b):
    ar = pd.Series(a).rank(method="average").to_numpy()
    br = pd.Series(b).rank(method="average").to_numpy()
    if np.std(ar) == 0.0 or np.std(br) == 0.0:
        return 0.0
    return float(np.corrcoef(ar, br)[0, 1])


def top_k_mask(values, k):
    order = np.argsort(-np.abs(values))
    mask = np.zeros(len(values), dtype=bool)
    mask[order[:k]] = True
    return mask


## The concept, built once (D1)

For an additive or locally linear model, SHAP-style attributions can be checked by conservation: $f(x)=\phi_0+\sum_{j=1}^{d}\phi_j$. We first build the exact additive receipt and assert the lesson's numbers.

In [ ]:

def explain_additive(model, x, background):
    baseline = np.mean(background, axis=0)
    phi0 = model(baseline)
    phi = np.zeros_like(x, dtype=float)
    for j in range(len(x)):
        moved = baseline.copy()
        moved[j] = x[j]
        phi[j] = model(moved) - model(baseline)
    return float(phi0), phi


def toy_shap_model(z):
    weights = np.array([1.2, 0.4, 0.3])
    return float(weights @ z)

x_toy = np.array([1.0, -1.0, 3.0])
background_toy = np.zeros((4, 3))
phi0_toy, phi_toy = explain_additive(toy_shap_model, x_toy, background_toy)
total_toy = float(phi_toy.sum())
abs_toy = float(np.abs(phi_toy).sum())
share_toy = float(np.max(np.abs(phi_toy)) / abs_toy)
guarded_toy = total_toy + 0.3 * abs_toy
contrast_toy = total_toy - phi_toy[2]

assert np.allclose(phi_toy, [1.2, -0.4, 0.9])
assert np.isclose(total_toy, 1.7)
assert np.isclose(abs_toy, 2.5)
assert np.isclose(share_toy, 0.48)
assert np.isclose(guarded_toy, 2.45)
assert np.isclose(contrast_toy - total_toy, -0.9)

print("phi0", phi0_toy)
print("phi", phi_toy)
print("signed total", total_toy)
print("absolute mass", abs_toy)
print("top share", share_toy)


The real ladder uses the same conservation idea, but the model is logistic regression. We approximate local SHAP by replacing one standardized feature at a time with its background mean and measure signed probability changes for the predicted class.

In [ ]:

def local_drop_attribution(model, background, x):
    class_index = int(np.argmax(model.predict_proba(x.reshape(1, -1))[0]))
    baseline = np.mean(background, axis=0)
    base_score = score_for_class(model, baseline.reshape(1, -1), class_index=class_index)[0]
    full_score = score_for_class(model, x.reshape(1, -1), class_index=class_index)[0]
    phi = np.zeros(x.shape[0])
    for j in range(x.shape[0]):
        hybrid = x.copy()
        hybrid[j] = baseline[j]
        hybrid_score = score_for_class(model, hybrid.reshape(1, -1), class_index=class_index)[0]
        phi[j] = full_score - hybrid_score
    residual = full_score - base_score - phi.sum()
    return base_score, full_score, phi, residual


def deletion_faithfulness(model, background, x):
    base_score, full_score, phi, residual = local_drop_attribution(model, background, x)
    drops = []
    magnitudes = []
    for j in range(len(phi)):
        hybrid = x.copy()
        hybrid[j] = np.mean(background, axis=0)[j]
        changed = score_for_class(model, hybrid.reshape(1, -1))[0]
        drops.append(full_score - changed)
        magnitudes.append(abs(phi[j]))
    return rank_corr(magnitudes, np.abs(drops)), phi, residual


## The dataset ladder

We use the shared F15 classification ladder: a hand linear toy, clean blobs, a nonlinear moons stress test, real Wine, and real Breast Cancer. The same logistic base model and the same metric code run unchanged across rungs.

In [ ]:

rungs = clf_ladder()

for index, item in enumerate(rungs, start=1):
    name, X, y = item
    counts = dict(zip(*np.unique(y, return_counts=True)))
    print(index, name)
    print("shape", X.shape)
    print("class counts", counts)
    print("sample", np.round(X[:3, : min(5, X.shape[1])], 3))


## Run the SAME method across D1-D5

The metric is faithfulness: do features with larger signed local attributions also cause larger deletion drops?

In [ ]:

shap_rows = []
shap_artifacts = []

for rung, item in enumerate(rungs, start=1):
    name, X, y = item
    model, scaler, x_tr, x_te, y_tr, y_te = fit_ladder_classifier(X, y)
    accuracy = clf_accuracy(lambda a, b, c: LogisticRegression(max_iter=2000).fit(a, b).predict(c), X, y)
    metric, phi, residual = deletion_faithfulness(model, x_tr, x_te[0])
    shap_rows.append({"rung": rung, "name": name, "accuracy": accuracy, "faithfulness": metric, "residual": residual})
    shap_artifacts.append((name, phi))

shap_table = pd.DataFrame(shap_rows)
print(shap_table.to_string(index=False))


## Results visualization

The left side shows signed local attributions per rung. The right side is the faithfulness-vs-stress curve from D1 to D5.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.ravel()

for ax, artifact in zip(axes[:5], shap_artifacts):
    name, phi = artifact
    keep = min(10, len(phi))
    colors = ["tab:blue" if value >= 0 else "tab:red" for value in phi[:keep]]
    ax.bar(np.arange(keep), phi[:keep], color=colors)
    ax.axhline(0.0, color="black", linewidth=0.8)
    ax.set_title(name[:26])
    ax.set_xlabel("feature")
    ax.set_ylabel("signed phi")

axes[5].plot(shap_table["rung"], shap_table["faithfulness"], marker="o")
axes[5].set_title("faithfulness vs stress")
axes[5].set_xlabel("rung")
axes[5].set_ylabel("rank correlation")
axes[5].set_ylim(-1.05, 1.05)
plt.tight_layout()


## Pitfall on the hardest rung

Pitfall: ranking by absolute value only hides whether a feature pushed the score up or down. The fix is to plot signed contributions against the background baseline.

In [ ]:

name, X, y = rungs[-1]
model, scaler, x_tr, x_te, y_tr, y_te = fit_ladder_classifier(X, y)
base_score, full_score, phi, residual = local_drop_attribution(model, x_tr, x_te[0])
absolute_order = np.argsort(-np.abs(phi))[:8]
signed_receipt = [(int(j), float(phi[j])) for j in absolute_order]
wrong_story = [(int(j), float(abs(phi[j]))) for j in absolute_order]

print("wrong absolute-only story", wrong_story)
print("fixed signed receipt", signed_receipt)
print("baseline", base_score)
print("full score", full_score)
print("signed sum plus baseline", base_score + phi.sum())


## Evaluate it + Practice

- Compare the reported deletion faithfulness with a no-skill baseline such as shuffled features, shuffled groups, or random rankings.
- Sanity check signs, denominators, and the background/reference point before trusting a pretty plot.
- Ablation: replace signed attributions with absolute ranks and watch the decision story lose direction
- Failure signals: unstable ranks under small perturbations, a metric that improves while accuracy collapses, or explanations that change when the seed changes.
- For gap topics, especially influence functions, keep the lesson numbers visible and treat the notebook as an audit scaffold until the lesson body grows more examples.

Practice 1: Change the trust knob or kernel width and predict which metric should move before running it.

Practice 2: Swap the D5 background/reference set and explain which conclusion is no longer stable.

Practice 3: Turn the pitfall back on, then write one sentence explaining why the fixed version is safer.